In [1]:
#Apply otliers removal
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
from sklearn.cluster import OPTICS
from sklearn.cluster import Birch

#load dataset
df= pd.read_csv("data/pain_dataset_200P_4hz.csv")
df

# Drop target and ID column & target column
X = df.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape:", X.shape)

Features shape: (96000, 7)


In [2]:
#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [3]:
# Compute z-scores
z_scores = np.abs((X - X.mean()) / X.std())

# Define threshold
threshold = 3
# Get row indices where ANY feature is an outlier
outlier_indices = np.where((z_scores > threshold).any(axis=1))[0]

# Remove them
X_clean = X.drop(index=X.index[outlier_indices])
print("Original features shape:", X.shape)
print("After Outlier Removal:", X_clean.shape)


Original features shape: (96000, 7)
After Outlier Removal: (95114, 7)


In [6]:
#Apply StandardScaler
scaler = StandardScaler()
X_so = scaler.fit_transform(X_clean)
print("Scaled shape:", X_so.shape)

Scaled shape: (95114, 7)


In [9]:
df_small = df.sample(n=5000, random_state=42)
X_small = df_small.drop(columns=["person_ID", "pain_scale"], errors="ignore")
# Compute z-scores
z_scores = np.abs((X_small - X_small.mean()) / X_small.std())

# Define threshold
threshold = 3
# Get row indices where ANY feature is an outlier
outlier_indices = np.where((z_scores > threshold).any(axis=1))[0]

# Remove them
X_small_clean = X_small.drop(index=X_small.index[outlier_indices])
X_small_out = scaler.fit_transform(X_small_clean)

print("Original Outlier Removal:", X_so.shape)
print("Original Outlier Removal:", X_small_clean.shape)
print("Small dataset Outlier Removal:", X_small_out.shape)


Original Outlier Removal: (95114, 7)
Original Outlier Removal: (4961, 7)
Small dataset Outlier Removal: (4961, 7)


In [10]:
#K-Means on Scaled + OutlierRemoval Data
start_time = time.time()
kmean_outliers = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    kmean_outliers.append({"algorithm": "KMeans", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"KMeans runtime: {runtime:.4f} seconds")   


Runtime: 1426.8165040016174 seconds
KMeans runtime: 1426.8165 seconds


In [11]:
#Gaussian Mixture (GMM)on Scaled + OutlierRemoval Data
start_time = time.time()
gmm_outliers = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_so)
    sil, db, ch = compute_metrics(X_so, labels)
    gmm_outliers.append({"algorithm": "GMM", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")   

Runtime: 2024.9298644065857 seconds
GMM runtime: 2024.9299 seconds


In [12]:
#Agglomerative Clustering on Scaled + OutlierRemoval Data
start_time = time.time()
agg_outliers = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_small_out)
    sil, db, ch = compute_metrics(X_small_out, labels)
    agg_outliers.append({"algorithm": "Agglomerative", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")   

Runtime: 13.385058403015137 seconds
Agglomerative runtime: 13.3851 seconds


In [13]:
#Spectral Clustering on Scaled + OutlierRemoval Data
start_time = time.time()
spec_outliers = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_small_out)
    sil, db, ch = compute_metrics(X_small_out, labels)
    spec_outliers.append({"algorithm": "Spectral", "preprocessing": "OutlierRemoval", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")   

Runtime: 36.65689468383789 seconds
Spectral runtime: 36.6569 seconds


In [14]:
#DBSCAN on Scaled + OutlierRemoval Data
start_time = time.time()
dbscan_outliers = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_so)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 and len(set(labels[mask])) > 1:
        sil, db, ch = compute_metrics(X_so[mask], labels[mask])
        dbscan_outliers.append({"algorithm": "DBSCAN", "preprocessing": "OutlierRemoval", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 368.0714075565338 seconds
DBSCAN runtime: 368.0714 seconds


In [15]:
#BIRCH on Scaled + OutlierRemoval Data
start_time = time.time()
birch_outliers = []
#threshold_values = [0.2, 0.5, 1.0, 1.5]
threshold_values = [1.5, 3.0, 5.0, 10.0]

for t in threshold_values:
    birch = Birch(n_clusters=None, branching_factor=200,threshold=t)
    labels = birch.fit_predict(X_so)

    n_clusters = len(set(labels))
    if 1 < n_clusters < len(X_so) :
        sil, db, ch = compute_metrics(X_so, labels)
        birch_outliers.append({
            "algorithm": "BIRCH",
            "preprocessing": "OutliersRemoval",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds")

Runtime: 187.5289487838745 seconds
BIRCH runtime: 187.5289 seconds


In [16]:
#OPTICS on Scaled + OutlierRemoval Data
start_time = time.time()
optics_outliers = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_so)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_so, labels)
        optics_outliers.append({
            "algorithm": "OPTICS",
            "preprocessing": "OutliersRemoval",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

Runtime: 15373.369999408722 seconds
Optics runtime: 15373.3700 seconds


In [17]:
import csv


breast_cancer_results_outliers = (kmean_outliers + gmm_outliers + agg_outliers + spec_outliers + dbscan_outliers+birch_outliers + optics_outliers)

keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

#with open('updated_data/pain_data/pain_outliers.csv', 'w', newline='') as file:
with open('updated_data/pain_new_data/pain_outliers.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(breast_cancer_results_outliers)

In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

# ARI stability analysis 
n_bootstrap = 100
ari_results = []
# Collect all parameter settings from your previous results
all_configs = []

for r in kmean_outliers:
    all_configs.append(("K-Means", {"k": r["k"]}))

for r in gmm_outliers:
    all_configs.append(("GMM", {"k": r["k"]}))

for r in agg_outliers:
    all_configs.append(("Agglomerative", {"k": r["k"]}))

for r in spec_outliers:
    all_configs.append(("Spectral", {"k": r["k"]}))

for r in dbscan_outliers:
    all_configs.append(("DBSCAN", {"eps": r["eps"]}))

for r in birch_outliers:
    all_configs.append(("BIRCH", {"threshold": r["threshold"]}))

for r in optics_outliers:
    all_configs.append(("OPTICS", {"min_samples": r["min_samples"]}))
# --- helper function to fit a model and return labels ---
def fit_and_predict(name, params, X_data):

    if name == "K-Means":
        model = KMeans(n_clusters=params["k"], n_init=n_init, random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "GMM":
        model = GaussianMixture(n_components=params["k"], n_init=n_init, random_state=42)
        labels = model.fit(X_data).predict(X_data)

    elif name == "Agglomerative":
        model = AgglomerativeClustering(n_clusters=params["k"], linkage='ward')
        labels = model.fit_predict(X_data)

    elif name == "Spectral":
        model = SpectralClustering(
            n_clusters=params["k"],
            affinity='nearest_neighbors',
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "DBSCAN":
        model = DBSCAN(eps=params["eps"], min_samples=min_samples)
        labels = model.fit_predict(X_data)

    elif name == "BIRCH":
        model = Birch(n_clusters=None, threshold=params["threshold"])
        labels = model.fit_predict(X_data)

    elif name == "OPTICS":
        model = OPTICS(min_samples=params["min_samples"], xi=0.05, n_jobs=-1)
        labels = model.fit_predict(X_data)

    else:
        return None

    return labels


# Reuse all parameter configurations from previous section
for algo_name, params in all_configs:

    # reference clustering on full data
    ref_labels = fit_and_predict(algo_name, params, X_so)

    if ref_labels is None:
        continue

    ari_scores = []
    rng = np.random.RandomState(42)

    for b in range(n_bootstrap):

        # bootstrap sample with indices
        indices = rng.choice(len(X_so), size=len(X_so), replace=True)
        X_boot = X_so[indices]

        boot_labels = fit_and_predict(algo_name, params, X_boot)

        if boot_labels is None:
            continue

        # compare only sampled observations
        ref_subset = np.array(ref_labels)[indices]

        # remove noise points for DBSCAN / OPTICS
        mask = (boot_labels != -1) & (ref_subset != -1)

        if np.sum(mask) < 2:
            continue

        ari = adjusted_rand_score(ref_subset[mask], np.array(boot_labels)[mask])
        ari_scores.append(ari)

    if len(ari_scores) > 0:
        ari_results.append({
            "algorithm": algo_name,
            **params,
            "ARI_mean": np.mean(ari_scores),
            "ARI_std": np.std(ari_scores)
        })


# Summary table
ari_df = pd.DataFrame(ari_results).round(4)

print("\n---- BOOTSTRAP ARI STABILITY ----")
print(ari_df.to_string(index=False))

# Top 3 most stable by ARI
top3_ari = ari_df.nlargest(3, "ARI_mean")

print("\n TOP 3 MOST STABLE BY ARI ")
print(top3_ari.to_string(index=False))

In [ ]:
#the top three by stability score
ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

top_3 = (
    ari_df
    .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
    .head(3)
    .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
)

print(top_3.round(3).to_string(index=False))

In [ ]:
#the best three clustering results overall, sort primarily by ARI_mean
top_3 = ari_df.sort_values(
    ["ARI_mean", "Stability Score"],
    ascending=[False, False]
).head(3)

print(top_3.round(3).to_string(index=False))

In [ ]:
ari_df.to_csv("updated_data/ARI_Score/pain_outliers_ari.csv", index=False)

In [ ]:
# Combine all algorithm results 
all_results = (
    kmean_outliers +
    gmm_outliers +
    agg_outliers +
    spec_outliers +
    dbscan_outliers +
    birch_outliers +
    optics_outliers
)

results_df = pd.DataFrame(all_results)

# Round metric values to 4 decimal places
metric_cols = ["silhouette", "davies_bouldin", "calinski_harabasz"]
results_df[metric_cols] = results_df[metric_cols].round(4)

# Columns that may exist depending on algorithm
possible_cols = ["algorithm", "k", "eps", "threshold", "min_samples","n_clusters"]

def available_cols(df, metric):
    cols = [c for c in possible_cols if c in df.columns]
    cols.append(metric)
    return cols

# Top 3 by Silhouette (higher is better) 
top3_sil = results_df.nlargest(3, "silhouette")

print("\n TOP 3 SILHOUETTE ")
print(top3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Top 3 by Davies-Bouldin (lower is better) 
top3_db = results_df.nsmallest(3, "davies_bouldin")

print("\n TOP 3 DAVIES-BOULDIN ")
print(top3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Top 3 by Calinski-Harabasz (higher is better) 
top3_ch = results_df.nlargest(3, "calinski_harabasz")

print("\n TOP 3 CALINSKI-HARABASZ ")
print(top3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))

# Bottom 3 by Silhouette (lower is worse) 
bottom3_sil = results_df.nsmallest(3, "silhouette")

print("\nBOTTOM 3 SILHOUETTE ")
print(bottom3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Bottom 3 by Davies-Bouldin (higher is worse) 
bottom3_db = results_df.nlargest(3, "davies_bouldin")

print("\n BOTTOM 3 DAVIES-BOULDIN ")
print(bottom3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

#  Bottom 3 by Calinski-Harabasz (lower is worse) 
bottom3_ch = results_df.nsmallest(3, "calinski_harabasz")

print("\n BOTTOM 3 CALINSKI-HARABASZ ")
print(bottom3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))


===== TOP 3 SILHOUETTE =====
algorithm   k  eps  threshold  min_samples  n_clusters  silhouette
   KMeans 2.0  NaN        NaN          NaN         NaN      0.2245
 Spectral 2.0  NaN        NaN          NaN         NaN      0.2180
      GMM 2.0  NaN        NaN          NaN         NaN      0.2133

===== TOP 3 DAVIES-BOULDIN =====
algorithm   k  eps  threshold  min_samples  n_clusters  davies_bouldin
   DBSCAN NaN  1.0        NaN          NaN         NaN          0.7893
   DBSCAN NaN  0.5        NaN          NaN         NaN          1.1379
   OPTICS NaN  NaN        NaN          3.0      8299.0          1.2066

===== TOP 3 CALINSKI-HARABASZ =====
algorithm   k  eps  threshold  min_samples  n_clusters  calinski_harabasz
   KMeans 2.0  NaN        NaN          NaN         NaN         32162.3079
      GMM 2.0  NaN        NaN          NaN         NaN         29992.7339
   KMeans 3.0  NaN        NaN          NaN         NaN         21855.3465

===== BOTTOM 3 SILHOUETTE =====
algorithm   k  eps

In [ ]:

# TOP 3 RESULTS FOR EACH ALGORITHM INDIVIDUALLY



all_algorithms = {
    "K-Means": kmean_outliers,
    "GMM": gmm_outliers,
    "Agglomerative": agg_outliers,
    "Spectral": spec_outliers,
    "DBSCAN": dbscan_outliers,
    "BIRCH": birch_outliers,
    "OPTICS": optics_outliers
}



for algorithm, results in all_algorithms.items():

    if len(results) == 0:
        continue

    result_df = pd.DataFrame(results)


    # Round all validation values to 4 decimal places

    result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ] = result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ].round(4)



    print("\n")
    print("="*60)
    print(algorithm)
    print("="*60)


    # Select parameter column


    parameter_columns = [
        "k",
        "eps",
        "threshold",
        "min_samples"
    ]


    parameter = None

    for col in parameter_columns:
        if col in result_df.columns:
            parameter = col
            break



    # TOP 3 SILHOUETTE
   

    print("\nTop 3 Silhouette Score (Higher is better)")

    top_sil = result_df.nlargest(
        3,
        "silhouette"
    )

    print(
        top_sil[
            [
                parameter,
                "silhouette"
            ]
        ].to_string(index=False)
    )



    # TOP 3 DAVIES-BOULDIN


    print("\nTop 3 Davies-Bouldin Index (Lower is better)")

    top_db = result_df.nsmallest(
        3,
        "davies_bouldin"
    )

    print(
        top_db[
            [
                parameter,
                "davies_bouldin"
            ]
        ].to_string(index=False)
    )



    # TOP 3 CALINSKI-HARABASZ
  

    print("\nTop 3 Calinski-Harabasz Index (Higher is better)")

    top_ch = result_df.nlargest(
        3,
        "calinski_harabasz"
    )

    print(
        top_ch[
            [
                parameter,
                "calinski_harabasz"
            ]
        ].to_string(index=False)
    )



K-Means

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.2245
 3      0.1507
 4      0.1399

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.6543
 8          1.8698
 7          1.9696

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2         32162.3079
 3         21855.3465
 4         17691.8535


GMM

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.2133
 3      0.1455
 4      0.1108

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.7087
 3          2.1406
 5          2.3928

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2         29992.7339
 3         20653.8659
 4         16079.2111


Agglomerative

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1809
 3      0.1011
 5      0.0778

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.8040
 3          2.0752
 8          2.3590

Top 3 Calinski-H